# Safari Compass Calibration

The Safari-Zone analog of the **Metronome Compass Calibration** notebook.  It
identifies the loaded seed, plans the manual advances needed to encounter a
Metang, identifies the *battle* seed from the safari encounter (bait / mud /
ball), checks how confident that identification is, saves the run, and feeds the
shared timer→frame calibration model.

## Two seeds, two kinds of "frame" (read this first)

Both seeds are fixed by **game-frame (clock) timing** — the timer precision we
calibrate:

- **Seed A** — the overworld stream: encounters, roamer relocation, Elm calls.
- **Seed B** — the battle stream: hits, crits, capture / flee odds.

An **advance frame** ("advance") is how many times a seed's state has been
advanced via `advance_rng`, driven by **player actions, not the clock**.  Section A
walks *Seed A's* advance frame (Elm calls + chatot flips + Sweet Scent) purely so
that we encounter a Metang — this does **not** affect Seed B or the calibration.
Calibration is the same timer(M)→Seed-B-frame fit as metronome; safari just
identifies Seed B differently and may carry a slightly different load-screen
offset, applied as a separate **safari offset** (β/slope stays from metronome).

## Sections
- **A** — identify Seed A (roamer + Elm), then plan the advances to a Metang.
- **B** — identify Seed B via safari compass, then a confidence / neighbor check.
- **C** — save the run to `data/safari_runs.jsonl`.
- **D** — analysis over the saved runs.
- **E** — apply the safari offset to `data/calibration_model.json` (offset only).

In [1]:
%load_ext autoreload
%autoreload 2
import datetime as dt

from utils.calibration_tools import (
    # Section A -- roamer routes + Elm seed identification (shared with metronome)
    generate_roamer_candidates_near,
    print_roamer_candidates,
    identify_seed,
    seed_for,                     # exact seed for a (datetime, delay) -- the intended target seed
    # Section C -- persist a safari run
    save_safari_run,
    # Section D -- analysis
    load_safari_runs,
    fit_safari_offset,
    # Section E -- apply the safari offset (deliberate; review-then-confirm)
    update_safari_offset,
)
from utils.safari_advance import (
    advance_context, context_from_row,
    identify_frame, prompt_target_frame, choose_target_frame,
    plan_advances, margin_guide, describe_plan,
)
from utils.safari_confidence import path_confidence, print_confidence

from claytonlib.compass import compass_safari, CompassSafariInput
from claytonlib.calibration import CalibrationModel
from claytonlib.safari import safari_pokemon_by_name
from claytonlib.chart import STRATEGY_ONLY_BALLS, CRITERIA_CAPTURE

## Section A — Identify Seed A, then plan to a Metang  (→ `a_seed`, advance plan)

Configure the target datetime/delay, the search window, and each roamer's **current**
route (before the reset).  After loading the save, this one cell:

1. **Identifies the seed** — roamer map + Elm phone (`identify_seed`): routes → Elm
   calls → (M) manual pick.
2. **Locates the current advance frame** from the Elm calls heard so far (1 Elm call =
   1 advance); the calls entered in step 1 are reused, prompting for more only if the
   frame is still ambiguous.
3. **Picks the target frame** (`choose_target_frame`):
   - Loaded the **target seed exactly** → `a_target_advances` (81 = shiny Metang),
     regardless of the toggle.
   - Else `a_use_inhouse=True` computes a Metang frame from the Safari block config,
     **skipping frames whose Elm-call margin is an ambiguous run** (e.g. `[KKK]`);
     `a_use_inhouse=False` falls back to the Pokefinder handoff.
   - `a_calibration_aim_advances` (**calibration aid only** — in-house, off-target): set it
     to aim for the Metang frame **nearest that advance** instead of the nearest to the
     current frame, so a data-gathering run can sweep a range of advances (how chatot flips /
     Elm calls affect `Fb`) rather than only low advance counts.  `None` = nearest-to-current
     (normal hunting).
   - `a_blocks` are block **scores** (day-multiplier applied): plains/forest/peak/water.
4. **Plans the advances** — chatot flips (2 each) for the bulk, then a verifiable Elm-call
   margin, then Sweet Scent (`]!` in the guide).  A lone half/single flip is folded into a
   4–5 call margin instead.

In [3]:
# --- Section A config ---
a_target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)   # <-- your load datetime
a_target_delay   = 681                                     # <-- your load delay
a_seconds_window = 1        # +/- X seconds
a_delay_window   = 60       # +/- Y delays
a_match_parity   = True     # only delays with target_delay's even/odd parity
a_display_limit  = 40       # rows to print (None = all)

# Each roamer's CURRENT route (before the reset).  A roamer roams iff it appears here.
a_prev_routes    = {"r": 30, "e": 32, "l": 5}

# Advance-planning config
a_use_inhouse     = True          # True = compute the Metang frame in-house; False = Pokefinder handoff
a_target_advances = 81            # frame to hit IF we loaded the target seed EXACTLY (shiny Metang)
a_calibration_aim_advances = 40   # CALIBRATION AID ONLY (in-house + off-target): aim for the Metang
                                  # frame nearest THIS advance to gather Fb data across a range of
                                  # advances; None = nearest to current frame (normal hunting)
a_area            = "Mountain"    # Safari area you're hunting in
a_tod             = "morning"     # time of day: morning / day / night
a_blocks          = {"peak": 56}  # block SCORES (day-multiplier applied): plains/forest/peak/water

# 1. Identify Seed A: roamer routes -> Elm calls -> (M) manual pick.
a_candidates = generate_roamer_candidates_near(
    a_target_time, a_target_delay, a_seconds_window, a_delay_window,
    prev_routes=a_prev_routes, match_parity=a_match_parity,
)
a_seed = identify_seed(a_candidates, display_limit=a_display_limit)

# 2. Locate the current advance frame (reusing the Elm calls already entered above; 1 call = 1
#    advance).  Prompts for more only if the frame is still ambiguous.
a_rng_calls, a_elm = advance_context(a_seed["seed"], a_prev_routes, count=160)
a_current_frame = identify_frame(a_rng_calls, a_elm,
                                 observed=a_seed.get("elm_observed", ""), max_offset=15)

# 3. Pick the target frame: exact-seed hit -> a_target_advances (any toggle); else the in-house
#    Metang frame (a_use_inhouse=True; skips ambiguous-margin frames) or the Pokefinder prompt.
#    a_calibration_aim_advances (if set) aims the in-house finder at the Metang frame nearest that
#    advance -- a calibration aid, not for normal hunting.
a_key_seed = seed_for(a_target_time, a_target_delay)
a_target_frame = choose_target_frame(
    a_seed["seed"], key_seed=a_key_seed, target_advances=a_target_advances,
    use_inhouse=a_use_inhouse, area=a_area, tod=a_tod, blocks=a_blocks,
    current_frame=a_current_frame, target="metang", rng_calls=a_rng_calls, elm=a_elm,
    aim_advance=a_calibration_aim_advances)

# 4. Plan the advances to the target frame.
a_plan  = plan_advances(a_current_frame, a_target_frame)
a_guide = margin_guide(a_rng_calls, a_elm, a_plan)
print(describe_plan(a_plan, a_guide))

Observed roamer routes (R E L, space-separated, . = any):  29 38 .



Observed R=29 E=38 L=.  ->  3 / 183 candidate(s) match

3 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C8  2025-07-24 14:45:54     687    +6   -1   29  38  19   3  PEEPKKPKKPEEPPP
  0x0C0E02C8  2025-07-24 14:45:55     687    +6   +0   29  38  19   3  KKPKKPKPPEKPKEE
  0x0D0E02C8  2025-07-24 14:45:56     687    +6   +1   29  38  19   3  PPKEKEPEEKPPEPE


Elm calls (type P/E/K as heard; M = pick manually):  pke


Elm calls so far: PKE
2 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0C0E02C8  2025-07-24 14:45:55     687    +6   +0   29  38  19   3  KKPKKPKPPEKPKEE
  0x0D0E02C8  2025-07-24 14:45:56     687    +6   +1   29  38  19   3  PPKEKEPEEKPPEPE


Elm calls (type P/E/K as heard; M = pick manually):  k


Elm calls so far: PKEK
1 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0D0E02C8  2025-07-24 14:45:56     687    +6   +1   29  38  19   3  PPKEKEPEEKPPEPE

=== Seed identified: 0x0D0E02C8  2025-07-24 14:45:56  delay=687  R/E/L=29/38/19  Elm=PPKEKEPEEKPPEPE ===
Elm calls: PKEK  ->  advance frame 8
In-house: metang at advance frame 41 (L44) -- closest to calibration aim advance 40; advance 33 from the current frame 8.
On advance frame 8; want a Metang encounter on frame 41 (Sweet Scent while on frame 41).
  Advances to go: 33
  1. 15 chatot flips (30 advances) -> land on frame 38
  2. 3 Elm calls -> frame 41, then Sweet Scent.
  Guide: PPKKK[KEP]!EEE   (]! = Sweet Scent here)


## Section B — Safari-compass Seed-B identification  (→ `b_matched`)

Drives the **calibrated** `compass_safari` (frame center from the model, ±kσ over
the RTC-second offsets), exactly as `expedition.compass_safari` does.  Walk the
safari encounter turn by turn — enter `m`/`b`/ball-shakes/`F`/`C` as you see them
— until the candidate set narrows.  Then a confidence check scans for other
nearby seeds that reproduce the same path (aliases), ranked by distance.

The boot key seed and initial time come straight from Section A's identified
`a_seed` (the loaded seed and its datetime) -- no need to re-enter them.  `b_M`
is the commanded countdown = `target_timer_delay + target_timer_calibration`.

In [4]:
# --- Section B: calibrated safari-compass target, then confidence / neighbor check ---
b_key_seed              = a_seed["seed"]                   # the loaded Seed A (from Section A)
b_initial_time          = a_seed["time"]                   # its datetime (from identify_seed)
b_target_timer_delay    = 249817                           # <-- commanded timer delay (ms)
b_target_timer_calibration = 0                             # <-- timer calibration (ms, signed)
b_max_target_seconds    = 600                              # <-- chart's max target (s)
b_pokemon_name          = "metang"
b_second_offsets        = (-1, 0, 1)   # cover off-by-one timer-start timing (the "3 seconds")
b_confidence_frame_range = 1000          # +/- frames to scan for path-aliases

b_M = b_target_timer_delay + b_target_timer_calibration
model = CalibrationModel.load_default()   # raw metronome fit (data/calibration_model.json)

b_inputs = CompassSafariInput.from_expedition_target(
    # Fold the fitted safari load-path offset into the frame center, exactly as
    # expedition.compass_safari does now -- otherwise Section B centers on the raw metronome
    # frame and drifts ~safari_offset frames off the expedition's target landing.  `model`
    # itself stays RAW so Section D's fit_safari_offset still measures against the metronome
    # fit (folding it there would collapse the offset to ~0 -- a double-correction).
    model=model.with_safari_offset(), M=b_M, initial_time=b_initial_time, key_seed=b_key_seed,
    max_target_seconds=b_max_target_seconds,
    pokemon=safari_pokemon_by_name(b_pokemon_name),
    strategy=STRATEGY_ONLY_BALLS, criteria=CRITERIA_CAPTURE,
    second_offsets=b_second_offsets, mass_cap=0.999,
)

# Interactive: enter the safari path as you play it out.
b_matched = compass_safari(b_inputs)

# Confidence / neighbor check: once a single seed remains, scan for aliases (other nearby seeds
# that reproduce the same path), ranked by distance.  The observed path comes straight from
# compass_safari -- no re-entry.
b_observed_path = b_matched.path
print(f"\nObserved path: {b_observed_path}")
if len(b_matched) == 1:
    b_seed = int(b_matched[0], 16)
    b_neighbors = path_confidence(b_inputs, b_seed, b_observed_path,
                                  frame_range=b_confidence_frame_range)
    print_confidence(b_neighbors)
else:
    b_seed = None
    print(f"{len(b_matched)} seeds still matched -- narrow further before trusting a single seed.")

=== Compass: Safari Zone Seed Identifier ===
  m      Mud, no crit                  Metang is angry!
  M / a  Mud, crit (Anger)             Metang is beside itself with anger!
  b      Bait, no crit                 Metang is eating!
  B / e  Bait, crit (Eating)           Metang is busy eating!
  0      Ball, 0 shakes                Oh, no! The Pokémon broke free!
  1      Ball, 1 shake                 Aww! It appeared to be caught!
  2      Ball, 2 shakes                Aargh! Almost had it!
  3      Ball, 3 shakes                Shoot! It was so close, too!
  C      Captured (ends)               Gotcha! Metang was caught!
  F      Fled (ends)                   Metang fled!
  u      Undo last action              —
  ?x     Uncertain result              —
  J      Switch to Jane                —
  w      Widen window & re-apply path  —
  Spaces and commas in input are ignored.


Seeds: 1093 / 1093 remaining
Path:  (none)
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1.


>>  b



Seeds: 985 / 1093 remaining
Path:  b
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xE50E3A6B    14955       0      0     0.51%
   2. 0xE50E3A6C    14956      +1      0     0.51%
   3. 0xE50E3A6A    14954      -1      0     0.51%
   4. 0xE50E3A6D    14957      +2      0     0.51%
   5. 0xE50E3A69    14953      -2      0     0.51%
  Most likely: 0xE50E3A6B  P=0.51%  (timer on time)



>>  b



Seeds: 677 / 1093 remaining
Path:  bb
Balls: 30
   #        Seed    Frame       Δ   δsec   P(land)
   1. 0xE50E3A6B    14955       0      0     0.73%
   2. 0xE50E3A6C    14956      +1      0     0.73%
   3. 0xE50E3A6A    14954      -1      0     0.73%
   4. 0xE50E3A6D    14957      +2      0     0.73%
   5. 0xE50E3A6E    14958      +3      0     0.73%
  Most likely: 0xE50E3A6B  P=0.73%  (timer on time)



>>  bF



Pokémon fled. 51 seed(s) matched this path:
Observed path: bbbF
  1. seed=0xE50E3A62  frame=14946  Δ=-9  δ=0s  P=9.19%
  2. seed=0xE50E3A61  frame=14945  Δ=-10  δ=0s  P=9.17%
  3. seed=0xE50E3A48  frame=14920  Δ=-35  δ=0s  P=7.77%
  4. seed=0xE50E3A47  frame=14919  Δ=-36  δ=0s  P=7.69%
  5. seed=0xE50E3A3A  frame=14906  Δ=-49  δ=0s  P=6.54%

Observed path: bbbF
51 seeds still matched -- narrow further before trusting a single seed.


## Section C — Save the run  (→ `data/safari_runs.jsonl`)

Appends this run — the identified Seed B (only when a single seed matched), the
Section-A `a_seed` (so the offset fit has `F_a`), the observed path, the commanded
timer (`b_target_timer_delay`, passed straight in), and the calibrated landing
(frame / RTC second / δ) — via the existing `save_safari_run`.  Prompts only for a
**tag** and **notes**, then confirms before writing (every safari run is a fresh
boot, so those fields are fixed).  Saving does **not** touch the calibration model
(that's Section E).

In [ ]:
# --- Section C: append this run to data/safari_runs.jsonl ---
# Uses b_target_timer_delay directly (no timer prompt); only prompts for tag, notes, and save.
# elm_calls / chatot_flips come from the Section A plan (recorded to study any correlation with
# the frame_delta / landing miss).
run_record = save_safari_run(b_matched, inputs=b_inputs, a_seed=a_seed, path=b_observed_path,
                             target_timer_delay=b_target_timer_delay,
                             elm_calls=a_plan.elm_before_scent, chatot_flips=a_plan.chatot_flips)

Run tag [KERCHAK]:  Calibrate Frames
Notes:  



{
  "saved_at": "2026-09-16T20:41:01",
  "tag": "Calibrate Frames",
  "target_timer_delay": 249817,
  "path": "bbbF",
  "n_matched": 51,
  "matched_seeds": [
    "0xE50E3A62",
    "0xE50E3A61",
    "0xE50E3A48",
    "0xE50E3A47",
    "0xE50E3A3A",
    "0xE50E3AA3",
    "0xE50E3A2D",
    "0xE50E3AB1",
    "0xE50E3A1F",
    "0xE50E3A12",
    "0xE40E3A6B",
    "0xE60E3A66",
    "0xE60E3A58",
    "0xE50E3A06",
    "0xE60E3A4B",
    "0xE60E3A8C",
    "0xE60E3A8D",
    "0xE60E3A3D",
    "0xE50E3AD8",
    "0xE50E3AD9",
    "0xE40E3AA0",
    "0xE40E3A36",
    "0xE60E3A31",
    "0xE40E3A2A",
    "0xE60E3A24",
    "0xE50E3AE6",
    "0xE40E3A1C",
    "0xE60E3A16",
    "0xE60E3AC1",
    "0xE60E3AC2",
    "0xE40E3AC7",
    "0xE50E3AF2",
    "0xE50E3AF3",
    "0xE60E3A09",
    "0xE60E3ACE",
    "0xE60E3ACF",
    "0xE50E39DE",
    "0xE40E39F5",
    "0xE50E39D1",
    "0xE50E39D0",
    "0xE40E3AEF",
    "0xE40E39E7",
    "0xE50E39C4",
    "0xE40E39DB",
    "0xE40E3AFC",
    "0xE60E39D4",
    "0xE60E3B

## Section D — Analysis over `safari_runs.jsonl`

Sparse for now.  Shows the run count and previews the safari **offset** the
current runs imply against the deployed model (does *not* write it).  The
safari-vs-metronome offset measurement proper is tracked in `clayton-abf.10`.

In [7]:
# --- Section D: quick look at the collected safari runs ---
runs = load_safari_runs()
confident = [r for r in runs if r.get("seed") is not None]
print(f"{len(runs)} safari run(s) saved; {len(confident)} with a confident single seed.")

fit = fit_safari_offset(model)   # holds the model slope; median residual = the offset
if fit:
    print(f"Safari offset preview: {fit['offset']:+.2f} frames "
          f"(n={fit['n']}, std={fit['std']:.2f})  -- not written until Section E.")
else:
    print("No usable runs yet (need a_seed + a confident single seed).")

18 safari run(s) saved; 14 with a confident single seed.
Safari offset preview: -407.64 frames (n=14, std=190.37)  -- not written until Section E.


## Section E — Apply the safari offset  (→ `data/calibration_model.json`)

Re-fits the safari **offset only** (holding the metronome slope/β) from
`safari_runs.jsonl`, shows the old → new offset per model, and writes it **only
after you confirm**.  It sets a *separate* `safari_offset` field — the metronome
`alpha`/`beta` are untouched.  The expedition folds this offset into the frame
center for **all** safari scoring (`use_safari_offset`, default True), and so does
Section B above, so after changing it **re-run `precompute_chart()`** (a fast
incremental extend to the shifted frames) **and then `chart_report()`**.

In [8]:
# --- Section E: review the safari offset re-fit, then write it only if confirmed ---
new_models = update_safari_offset()


[linear] safari_offset -435.27 -> -407.64 frames  (n=14, std=190.37)
[quad] safari_offset -460.97 -> -414.04 frames  (n=14, std=202.13)

*** This shifts the safari load-path frame center. The expedition applies it to ALL safari scoring (use_safari_offset, default True), so RE-RUN precompute_chart() to extend the canon to the shifted frames, then chart_report(). (Set expedition.use_safari_offset=False to score against the raw metronome fit instead.) ***


Write the safari offset? (y/n):  n


Safari offset not updated.
